# 02 — Video Branch: CNN-LSTM
### Phase 2 of the implementation plan

**Revised pipeline (GPU-safe):**
1. **Cell 2 — One-time frame extraction** (run once, CPU): OpenCV decodes each video and saves 16 pre-processed frames as a `.pt` tensor to `data/video_frames/`. Once done, OpenCV is never touched again.
2. **Cell 3+ — Training** (GPU only): Dataset loads `.pt` files directly — no video decoding, no CPU bottleneck during training.

Corresponds to: `Video → Frame Extraction → CNN → LSTM → Video Feature Vector` in the project design.

In [ ]:
import os, json, random
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision.models as tvm

# Throttle CPU threads — training is GPU-only, no need for CPU parallelism
torch.set_num_threads(2)

with open("shared_config.json") as f:
    CFG = json.load(f)

PROJECT_ROOT = CFG["project_root"]
VID_CFG      = CFG["video"]
SEED         = CFG["seed"]

NUM_FRAMES  = VID_CFG["num_frames"]   # 16
FRAME_SIZE  = VID_CFG["frame_size"]   # 224
FRAMES_DIR  = os.path.join(PROJECT_ROOT, "data", "video_frames")

random.seed(SEED)
torch.manual_seed(SEED)

try:
    import torch_directml
    device = torch_directml.device()
    print("Device: DirectML GPU (AMD Radeon RX 580)", device)
except ImportError:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Device:", device)

## 1. One-time frame extraction (CPU — run once only)

Decodes every video with OpenCV and saves each as a pre-processed `(16, 3, 224, 224)` float32 tensor.
Output goes to `data/video_frames/real/` and `data/video_frames/fake/`.

**Already-extracted files are skipped automatically** — safe to re-run.

In [ ]:
import cv2
import numpy as np
import torchvision.transforms as T
import sys, time

frame_tf = T.Compose([
    T.ToPILImage(),
    T.Resize((FRAME_SIZE, FRAME_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std =[0.229, 0.224, 0.225]),
])

def extract_and_save_frames(video_path, out_path, num_frames=NUM_FRAMES):
    """Decode video with OpenCV, save tensor to out_path. Returns True on success."""
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        cap.release()
        return False
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total <= 0:
        cap.release()
        return False

    indices = np.linspace(0, max(total - 1, 0), num_frames).astype(int)
    frames = []
    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if ret and frame is not None:
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frames.append(frame_tf(frame))
    cap.release()

    if len(frames) == 0:
        return False
    while len(frames) < num_frames:
        frames.append(frames[-1])
    tensor = torch.stack(frames[:num_frames])  # (16, 3, 224, 224) float32
    torch.save(tensor, out_path)
    return True

# ---- Run extraction ----
video_root = os.path.join(PROJECT_ROOT, "data", "video")
os.makedirs(FRAMES_DIR, exist_ok=True)

total_saved = 0
total_skipped = 0
total_failed = 0
t0 = time.time()

for cls in ["real", "fake"]:
    src_dir = os.path.join(video_root, cls)
    dst_dir = os.path.join(FRAMES_DIR, cls)
    os.makedirs(dst_dir, exist_ok=True)

    videos = [f for f in os.listdir(src_dir) if f.lower().endswith((".mp4", ".avi", ".mov"))]
    n = len(videos)

    for i, fname in enumerate(videos, 1):
        stem   = os.path.splitext(fname)[0]
        out_pt = os.path.join(dst_dir, stem + ".pt")

        if os.path.exists(out_pt):          # already done — skip
            total_skipped += 1
        else:
            ok = extract_and_save_frames(os.path.join(src_dir, fname), out_pt)
            if ok:
                total_saved += 1
            else:
                total_failed += 1

        elapsed = time.time() - t0
        done    = total_saved + total_skipped + total_failed
        speed   = done / elapsed if elapsed > 0 else 0
        sys.stdout.write(f"\r[{cls}] {i}/{n} | saved={total_saved} skipped={total_skipped} failed={total_failed} | {speed:.1f} vid/s")
        sys.stdout.flush()

    sys.stdout.write("\n")

print(f"\nDone. Saved={total_saved} | Skipped={total_skipped} | Failed={total_failed}")
print(f"Frames stored in: {FRAMES_DIR}")

## 2. Dataset — loads pre-extracted .pt tensors (no OpenCV, no CPU decoding)

Expects:
```
data/video_frames/real/*.pt
data/video_frames/fake/*.pt
```
Each `.pt` is a `(16, 3, 224, 224)` float32 tensor already normalised.

In [ ]:
class VideoFrameDataset(Dataset):
    """Loads pre-extracted .pt frame tensors. Zero OpenCV / zero video decoding."""
    def __init__(self, frames_root):
        self.samples = []
        for label, cls in enumerate(["real", "fake"]):
            cls_dir = os.path.join(frames_root, cls)
            if os.path.isdir(cls_dir):
                for fname in os.listdir(cls_dir):
                    if fname.endswith(".pt"):
                        self.samples.append((os.path.join(cls_dir, fname), label))
        if len(self.samples) == 0:
            raise RuntimeError(f"No .pt files found under {frames_root}. Run the frame extraction cell first.")
        print(f"Dataset: {len(self.samples)} pre-extracted samples found.")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        frames = torch.load(path, weights_only=True)   # (16, 3, 224, 224) float32
        return frames, label


full_dataset = VideoFrameDataset(FRAMES_DIR)

val_len   = max(1, int(0.2 * len(full_dataset)))
train_len = len(full_dataset) - val_len
train_ds, val_ds = random_split(full_dataset, [train_len, val_len])

# num_workers=0: tensors load fast from disk; extra workers add overhead not savings here
train_loader = DataLoader(train_ds, batch_size=VID_CFG["batch_size"], shuffle=True,  num_workers=0, pin_memory=False)
val_loader   = DataLoader(val_ds,   batch_size=VID_CFG["batch_size"], shuffle=False, num_workers=0, pin_memory=False)

print(f"Train samples: {len(train_ds)} | Val samples: {len(val_ds)}")

## 3. Model — CNN (per-frame) + DirectML-compatible LSTM (temporal)

ResNet-18 backbone (frozen) extracts per-frame spatial features; a pure-PyTorch LSTM unrolled cell models temporal relationships across the frame sequence on the DirectML GPU.

In [ ]:
class DirectMLLSTM(nn.Module):
    """Pure PyTorch LSTM cell — runs 100% on DirectML, no fused C++ ops."""
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.gates = nn.Linear(input_size + hidden_size, 4 * hidden_size)

    def forward(self, x):
        B, T, C = x.shape
        h = torch.zeros(B, self.hidden_size, device=x.device)
        c = torch.zeros(B, self.hidden_size, device=x.device)
        for t in range(T):
            combined = torch.cat([x[:, t, :], h], dim=1)
            gates    = self.gates(combined)
            i, f, g, o = gates.chunk(4, dim=1)
            i, f, o = torch.sigmoid(i), torch.sigmoid(f), torch.sigmoid(o)
            g = torch.tanh(g)
            c = f * c + i * g
            h = o * torch.tanh(c)
        return h


class CNNLSTMDetector(nn.Module):
    def __init__(self, lstm_hidden=VID_CFG["lstm_hidden"], num_classes=2):
        super().__init__()
        resnet = tvm.resnet18(weights=tvm.ResNet18_Weights.IMAGENET1K_V1)
        self.cnn = nn.Sequential(*list(resnet.children())[:-1])  # drop FC → (B, 512, 1, 1)
        for p in self.cnn.parameters():
            p.requires_grad = False          # freeze CNN backbone entirely

        self.cnn_out_dim = resnet.fc.in_features   # 512
        self.lstm        = DirectMLLSTM(self.cnn_out_dim, lstm_hidden)
        self.feature_dim = lstm_hidden
        self.classifier  = nn.Sequential(
            nn.Linear(lstm_hidden, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def extract_features(self, x):
        """x: (B, T, C, H, W) -> F_video: (B, lstm_hidden).
        CNN is frozen (requires_grad=False) — no no_grad() needed.
        Per-frame processing avoids DirectML heap buffer overflows."""
        B, T, C, H, W = x.shape
        feats = [self.cnn(x[:, t]).flatten(1) for t in range(T)]  # list of (B, 512)
        return self.lstm(torch.stack(feats, dim=1))                # (B, lstm_hidden)

    def forward(self, x):
        return self.classifier(self.extract_features(x))


model = CNNLSTMDetector().to(device)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)")
print(f"Feature vector dimension (F_video): {model.feature_dim}")

## 4. Train (GPU only — LSTM + classifier, CNN frozen)

In [ ]:
import sys, time

EPOCHS = 3
LR     = 1e-2   # higher LR suits SGD

criterion = nn.CrossEntropyLoss()
# SGD+momentum is fully supported on DirectML — no CPU fallback unlike Adam's lerp step
optimizer = torch.optim.SGD(
    [p for p in model.parameters() if p.requires_grad],
    lr=LR, momentum=0.9, weight_decay=1e-4
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

def run_epoch(epoch, loader, train=True):
    model.train() if train else model.eval()
    total_loss, correct, n = 0.0, 0, 0
    total_items   = len(loader.dataset)
    total_batches = len(loader)
    mode          = "Train" if train else "Val"
    t0            = time.time()

    with torch.set_grad_enabled(train):
        for batch_idx, (frames, labels) in enumerate(loader, 1):
            frames, labels = frames.to(device), labels.to(device)
            outputs = model(frames)
            loss    = criterion(outputs, labels)

            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            bs          = frames.size(0)
            total_loss += loss.item() * bs
            correct    += (outputs.argmax(1) == labels).sum().item()
            n          += bs

            elapsed   = time.time() - t0
            speed     = n / elapsed if elapsed > 0 else 0
            remaining = (total_items - n) / speed if speed > 0 else 0
            el_m, el_s  = divmod(int(elapsed),   60)
            rem_m, rem_s = divmod(int(remaining), 60)
            pct = (n / total_items) * 100

            sys.stdout.write(
                f"\rEpoch {epoch}/{EPOCHS} [{mode}] | "
                f"Batch {batch_idx}/{total_batches} ({n}/{total_items}, {pct:.1f}%) | "
                f"Loss: {total_loss/n:.4f} | Acc: {correct/n:.3f} | "
                f"{speed:.1f} vid/s | [{el_m:02d}:{el_s:02d}<{rem_m:02d}:{rem_s:02d}]"
            )
            sys.stdout.flush()

    sys.stdout.write("\n")
    sys.stdout.flush()
    return total_loss / n, correct / n

print("Starting CNN-LSTM Training (GPU only)...", flush=True)
for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = run_epoch(epoch, train_loader, train=True)
    val_loss,   val_acc   = run_epoch(epoch, val_loader,   train=False)
    scheduler.step()
    print(
        f"--> Epoch {epoch}/{EPOCHS} Complete | "
        f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.3f} | "
        f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.3f}\n",
        flush=True
    )

## 5. Save checkpoint

In [ ]:
ckpt_path  = os.path.join(PROJECT_ROOT, "checkpoints", "cnn_lstm_video.pt")
model_path = os.path.join(PROJECT_ROOT, "models",      "cnn_lstm_video.pt")
payload = {"model_state_dict": model.state_dict(), "feature_dim": model.feature_dim}
torch.save(payload, ckpt_path)
torch.save(payload, model_path)
print("Saved:", ckpt_path)
print("Saved:", model_path)

## 6. Sanity check — extract F_video for a single sample

In [ ]:
model.eval()
sample_frames, sample_label = full_dataset[0]
with torch.no_grad():
    F_video = model.extract_features(sample_frames.unsqueeze(0).to(device))

print("F_video shape :", tuple(F_video.shape))          # expect (1, 256)
print("F_video[0,:10]:", F_video[0, :10].cpu().numpy())

## Done — Video branch is ready

You now have:
- Pre-extracted frames in `data/video_frames/` (load once, reuse forever)
- A fine-tuned CNN-LSTM checkpoint at `checkpoints/cnn_lstm_video.pt`
- A `model.extract_features(x)` method producing `F_video` (256-dim)

Next: open `03_audio_wavlm.ipynb` for the audio branch.